# 04 — Risk Scoring and Decisions

This notebook loads the complete scorecard bundle and creates transaction-level risk decisions. No separate new CSV is required for the demonstration.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from src.scorecard import load_bundle, load_creditcard_csv, score_transactions

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
MODEL_PATH = PROJECT_ROOT / 'model' / 'scorecard_bundle.pkl'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'scored_transactions_demo.csv'
FIGURE_DIR = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
bundle = load_bundle(MODEL_PATH)
print('Bundle version:', bundle['version'])
print('Features:', bundle['features'])

In [ ]:
df = load_creditcard_csv(DATA_PATH)
normal_demo = df[df['Class'] == 0].sample(950, random_state=42)
fraud_demo = df[df['Class'] == 1].sample(min(50, int((df['Class'] == 1).sum())), random_state=42)
demo = pd.concat([normal_demo, fraud_demo], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print('Demo shape:', demo.shape)
demo['Class'].value_counts()

In [ ]:
scored = score_transactions(demo.drop(columns='Class'), bundle)
scored.insert(0, 'Transaction_ID', range(1, len(scored) + 1))
scored['Actual_Class'] = demo['Class'].to_numpy()
scored = scored.sort_values('Risk_Score').reset_index(drop=True)
scored.head(10)[['Transaction_ID', 'Amount', 'Fraud_Probability', 'Risk_Score', 'Risk_Level', 'Recommended_Action', 'Actual_Class']]

In [ ]:
scored.to_csv(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)

In [ ]:
decision_summary = scored.groupby(['Risk_Level', 'Recommended_Action'], observed=False).agg(
    Transactions=('Transaction_ID', 'count'),
    Fraud_Count=('Actual_Class', 'sum'),
    Average_Score=('Risk_Score', 'mean')
).reset_index()
decision_summary['Fraud_Rate'] = decision_summary['Fraud_Count'] / decision_summary['Transactions'] * 100
decision_summary

In [ ]:
order = ['High Risk', 'Medium Risk', 'Low Risk']
ax = sns.barplot(data=decision_summary, x='Risk_Level', y='Fraud_Rate', order=order, color='#E45756')
ax.set(title='Fraud Rate by Risk Level', xlabel='Risk Level', ylabel='Fraud Rate (%)')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'score_band_fraud_rate.png', dpi=200, bbox_inches='tight')
plt.show()

## Operational interpretation

Low scores represent high estimated fraud risk. Scores 0–300 are blocked, 301–500 are sent to manual review, and scores above 500 are approved. These thresholds should be adjusted using investigation capacity, false-positive cost and fraud-loss estimates before production use.